In [1]:
#==============================================================================
# Loading the data
#==============================================================================

import pandas as pd

url = "https://raw.githubusercontent.com/Amirhossein-momeni/AmirGeek_ML_Datasets/refs/heads/main/DataSet/01-Ad.csv"
df = pd.read_csv(url)

In [2]:
#==============================================================================
# Exploring the data (EDA)
#==============================================================================

print(70* "-")  
print(df.shape) 
print(70* "-")                                     
print(df.head())
print(70* "-")  
print(df.info())
print(70* "-")  
print(df.isna().sum())
print(70* "-")    
print(df.isna().sum().sum()) 
print(70* "-")                              
print(df["Purchased"].value_counts(normalize=True))  
print(70* "-")  

----------------------------------------------------------------------
(400, 6)
----------------------------------------------------------------------
   UserID  Gender Type  Age  EstimatedSalary  Purchased
0       1    Male    B   24          2800000          0
1       2    Male    A   48          3000000          0
2       3  Female    B   34          7800000          0
3       4  Female    B   36         10700000          0
4       5    Male    D   24         14700000          0
----------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   UserID           400 non-null    int64 
 1   Gender           400 non-null    object
 2   Type             400 non-null    object
 3   Age              400 non-null    int64 
 4   EstimatedSalary  400 non-null    int64 
 5   Purchased        4

In [3]:
#==============================================================================
# Defining features and target
#==============================================================================

X = df.drop(columns=["UserID", "Purchased"])
y = df["Purchased"]

In [4]:
#==============================================================================
# Train / Test split
#==============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
#==============================================================================
# Preprocessing (two candidate feature sets)
#==============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["Age", "EstimatedSalary"]
cat_cols = ["Gender", "Type"]

prep_num_only = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
])

prep_with_cat = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols),
])

In [6]:
#==============================================================================
# Building the model + choosing the feature set with CV
#==============================================================================

from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

model_num = Pipeline([("prep", prep_num_only), ("knn", KNeighborsClassifier(n_neighbors=5))])
model_cat = Pipeline([("prep", prep_with_cat), ("knn", KNeighborsClassifier(n_neighbors=5))])

for name, m in [("numeric only", model_num), ("numeric + categorical", model_cat)]:
    score = cross_val_score(m, X_train, y_train, cv=5, scoring="f1").mean()
    print(name, round(score, 3))
# numeric only          -> 0.875
# numeric + categorical -> 0.837

numeric only 0.875
numeric + categorical 0.837


In [7]:
#==============================================================================
# Experiment: what happens WITHOUT scaling?
#==============================================================================

from sklearn.metrics import accuracy_score, confusion_matrix

prep_raw = ColumnTransformer([("num", "passthrough", num_cols)])
model_raw = Pipeline([("prep", prep_raw), ("knn", KNeighborsClassifier(n_neighbors=5))])

model_raw.fit(X_train, y_train)
y_pred_raw = model_raw.predict(X_test)

print(accuracy_score(y_test, y_pred_raw))      # 0.7625
print(confusion_matrix(y_test, y_pred_raw))    # [[46  5] [14 15]]
print(cross_val_score(model_raw, X_train, y_train, cv=5, scoring="f1").mean())  # ~0.72

0.7625
[[46  5]
 [14 15]]
0.7149182377272689


In [8]:
#==============================================================================
# How does k affect overfitting? (train accuracy vs CV score)
#==============================================================================

for k in [1, 3, 5, 7, 11, 15, 21, 31, 51]:
    m = Pipeline([("prep", prep_num_only), ("knn", KNeighborsClassifier(n_neighbors=k))])
    cv_f1 = cross_val_score(m, X_train, y_train, cv=5, scoring="f1").mean()
    train_acc = m.fit(X_train, y_train).score(X_train, y_train)
    print(k, round(train_acc, 3), round(cv_f1, 3))

1 0.994 0.823
3 0.925 0.857
5 0.922 0.875
7 0.925 0.879
11 0.922 0.869
15 0.912 0.868
21 0.9 0.865
31 0.9 0.862
51 0.897 0.784


In [9]:
#==============================================================================
# Training + evaluation (baseline, k=5)
#==============================================================================

from sklearn.metrics import classification_report

model = model_num
model.fit(X_train, y_train)          # KNN "training" = just storing the data

y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))          # 0.8875
print(confusion_matrix(y_test, y_pred))        # [[46  5] [ 4 25]]
print(classification_report(y_test, y_pred))

0.8875
[[46  5]
 [ 4 25]]
              precision    recall  f1-score   support

           0       0.92      0.90      0.91        51
           1       0.83      0.86      0.85        29

    accuracy                           0.89        80
   macro avg       0.88      0.88      0.88        80
weighted avg       0.89      0.89      0.89        80



In [10]:
#==============================================================================
# Looking inside: who are the neighbors of one customer?
#==============================================================================

one_customer = X_test.iloc[[0]]                       # Female, type A, age 68, salary 23.6M
Z = model.named_steps["prep"].transform(one_customer)  # scale the same way as training
distances, idx = model.named_steps["knn"].kneighbors(Z)

print(y_train.iloc[idx[0]].tolist())    # labels of the neighbors

[1, 1, 1, 1, 1]


In [11]:
#==============================================================================
# Hyperparameter tuning
#==============================================================================

from sklearn.model_selection import GridSearchCV

param_grid = {
    "knn__n_neighbors": list(range(1, 32, 2)),   # odd k avoids ties in binary problems
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

search = GridSearchCV(model_num, param_grid, cv=5, scoring="f1", n_jobs=-1)
search.fit(X_train, y_train)

print(search.best_params_)   # {'knn__n_neighbors': 7, 'knn__p': 1, 'knn__weights': 'uniform'}
print(search.best_score_)    # ~0.879

{'knn__n_neighbors': 7, 'knn__p': 1, 'knn__weights': 'uniform'}
0.879304347826087


In [12]:
#==============================================================================
# Final evaluation (once!) + probabilities
#==============================================================================

best_model = search.best_estimator_
y_pred_final = best_model.predict(X_test)

print(accuracy_score(y_test, y_pred_final))      # 0.925
print(confusion_matrix(y_test, y_pred_final))    # [[48  3] [ 3 26]]
print(classification_report(y_test, y_pred_final))

proba = best_model.predict_proba(X_test)[:, 1]
print(proba[:3].round(3))                        # [0.857 0.    0.   ]
print(y_test.iloc[:3].tolist())                  # [1, 0, 0]

0.925
[[48  3]
 [ 3 26]]
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        51
           1       0.90      0.90      0.90        29

    accuracy                           0.93        80
   macro avg       0.92      0.92      0.92        80
weighted avg       0.93      0.93      0.93        80

[0.857 0.    0.   ]
[1, 0, 0]
